# P1 · Data audit, cleaning and split
**Owner: Member A** · CMPS 261 Project 4.1 · Kepler Objects of Interest

What this notebook does:
1. Loads the full KOI cumulative table from the NASA Exoplanet Archive (downloads it the first time).
2. Audits it: class counts, how many KOIs share a star, % missing per feature.
3. Cleans it: drops duplicate `kepoi_name` and rows with non-positive period, duration, depth, planet radius or star radius. Missing values are **not** filled here (that happens inside the pipeline in P2).
4. Keeps CONFIRMED (label 1) and FALSE POSITIVE (label 0); saves CANDIDATE rows separately.
5. Lazy baseline: "everything is a false positive".
6. Splits 60 / 20 / 20 **by star** (`kepid`), checks no star is in two sets, and saves `data/processed/koi_split.csv`.

Outputs (shared names from the plan): `data/processed/koi_split.csv`, `data/processed/koi_candidates.csv`.

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Works whether the notebook runs from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "koi_cumulative.csv"
PROCESSED = ROOT / "data" / "processed"
RAW.parent.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

SEED = 42
pd.set_option("display.max_rows", 80)
print("Repo root:", ROOT)

Repo root: c:\Users\Administrator\kepler-koi-classification


## 1. Load the data
The first run downloads **all columns** of the `cumulative` table through the archive's TAP service and saves it to `data/raw/koi_cumulative.csv`, plus the download date (needed for the README). Later runs reuse the saved file, so both members work on the same snapshot.

If the download fails (no internet / SSL error): open the [KOI cumulative table](https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=cumulative), choose **Download Table → CSV format, All Columns**, and save it as `data/raw/koi_cumulative.csv`. The loader below skips the `#` comment lines that the website adds.

In [14]:
URL = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
       "?query=select+*+from+cumulative&format=csv")

if not RAW.exists():
    print("Downloading the KOI cumulative table ...")
    pd.read_csv(URL, low_memory=False).to_csv(RAW, index=False)
    (RAW.parent / "DOWNLOAD_DATE.txt").write_text(str(pd.Timestamp.today().date()))

# Skip '#' comment lines at the top (present when downloaded from the website)
n_comment = 0
with open(RAW, encoding="utf-8") as f:
    for line in f:
        if not line.startswith("#"):
            break
        n_comment += 1
df = pd.read_csv(RAW, skiprows=n_comment, low_memory=False)

date_file = RAW.parent / "DOWNLOAD_DATE.txt"
print("Download date:", date_file.read_text() if date_file.exists() else "unknown (manual download: write the date in DOWNLOAD_DATE.txt)")
print("Shape:", df.shape)

Download date: 2026-11-19
Shape: (9564, 153)


In [15]:
# Columns the project needs (plan section 2). If any is missing you probably have the Kaggle copy.
TRANSIT  = ["koi_period", "koi_duration", "koi_depth", "koi_impact", "koi_ror",
            "koi_model_snr", "koi_num_transits"]
PLANET   = ["koi_prad", "koi_teq", "koi_insol"]
STAR     = ["koi_steff", "koi_slogg", "koi_srad", "koi_smet", "koi_kepmag"]
DIAG     = ["koi_dicco_msky", "koi_dikco_msky", "koi_bin_oedp_sig"]
EXTRA    = ["koi_srho"]                      # needed for the engineered density_ratio (P2)
FEATURES = TRANSIT + PLANET + STAR + DIAG
IDS      = ["kepid", "kepoi_name", "koi_disposition"]

missing_cols = [c for c in IDS + FEATURES + EXTRA if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}. Download the full table from the NASA archive."
print(f"All {len(IDS + FEATURES + EXTRA)} required columns are present.")

All 22 required columns are present.


## 2. Audit (find problems first, R3)

In [16]:
# Class counts
counts = df["koi_disposition"].value_counts()
audit_classes = pd.DataFrame({"count": counts, "%": (100 * counts / len(df)).round(1)})
audit_classes

,count,%
koi_disposition,,
FALSE POSITIVE,4839,50.6
CONFIRMED,2748,28.7
CANDIDATE,1977,20.7


In [17]:
# How many KOIs share a star?
per_star = df.groupby("kepid").size()
multi = per_star[per_star > 1]
print(f"KOIs: {len(df)}  |  stars: {per_star.size}")
print(f"Stars with more than one KOI: {multi.size} ({100 * multi.size / per_star.size:.1f}% of stars)")
print(f"KOIs that live on a multi-KOI star: {multi.sum()} ({100 * multi.sum() / len(df):.1f}% of KOIs)")
print(f"Most KOIs on one star: {per_star.max()}")
per_star.value_counts().sort_index().rename_axis("KOIs per star").to_frame("number of stars")

KOIs: 9564  |  stars: 8214
Stars with more than one KOI: 938 (11.4% of stars)
KOIs that live on a multi-KOI star: 2288 (23.9% of KOIs)
Most KOIs on one star: 7


,number of stars
KOIs per star,
1,7276
2,661
3,179
4,68
5,25
6,3
7,2


In [18]:
# % missing per feature (sorted)
missing_pct = (100 * df[FEATURES + EXTRA].isna().mean()).round(2).sort_values(ascending=False)
missing_pct.to_frame("% missing")

,% missing
koi_bin_oedp_sig,15.79
koi_num_transits,11.94
koi_dicco_msky,6.26
koi_dikco_msky,5.96
koi_smet,4.04
koi_depth,3.80
koi_steff,3.80
koi_slogg,3.80
koi_impact,3.80
koi_ror,3.80


## 3. Clean
Drop duplicate `kepoi_name` and physically impossible rows (≤ 0 in period, duration, depth, planet radius or star radius). Rows where these are **missing** are kept: missing values are imputed later inside the pipeline, using training rows only.

In [19]:
n_start = len(df)
df = df.drop_duplicates(subset="kepoi_name", keep="first")
n_dup = n_start - len(df)

MUST_BE_POSITIVE = ["koi_period", "koi_duration", "koi_depth", "koi_prad", "koi_srad"]
nonpos = df[MUST_BE_POSITIVE] <= 0          # NaN <= 0 is False, so missing values are kept
print("Rows with a value <= 0, per column:")
print(nonpos.sum().to_string())

df = df[~nonpos.any(axis=1)].copy()
print(f"\nDropped {n_dup} duplicate rows and {n_start - n_dup - len(df)} rows with values <= 0.")
print(f"Rows left: {len(df)}")

Rows with a value <= 0, per column:
koi_period      0
koi_duration    0
koi_depth       1
koi_prad        0
koi_srad        0

Dropped 0 duplicate rows and 1 rows with values <= 0.
Rows left: 9563


## 4. Labels
CONFIRMED → 1, FALSE POSITIVE → 0. CANDIDATE rows have no known answer, so they are set aside (ranked with the final model in P7, optional).

In [20]:
candidates = df[df["koi_disposition"] == "CANDIDATE"].copy()
lab = df[df["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])].copy()
lab["label"] = (lab["koi_disposition"] == "CONFIRMED").astype(int)
lab = lab.reset_index(drop=True)

other = df.loc[~df["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"]), "koi_disposition"]
if len(other):
    print("Other dispositions dropped:", other.value_counts().to_dict())

candidates.to_csv(PROCESSED / "koi_candidates.csv", index=False)
print(f"Labelled rows: {len(lab)}  (planets: {lab.label.sum()}, false positives: {(lab.label == 0).sum()})")
print(f"Planet rate: {lab.label.mean():.3f}")
print(f"CANDIDATE rows saved separately: {len(candidates)}")

Labelled rows: 7587  (planets: 2748, false positives: 4839)
Planet rate: 0.362
CANDIDATE rows saved separately: 1976


## 5. Lazy baseline (R3.1)
Predict "false positive" for everything. Accuracy looks decent only because of class imbalance; F1 for the planet class is 0. This is why F1 is the main metric.

In [21]:
y = lab["label"].to_numpy()
y_lazy = np.zeros_like(y)
baseline = pd.Series({
    "accuracy":  accuracy_score(y, y_lazy),
    "precision": precision_score(y, y_lazy, zero_division=0),
    "recall":    recall_score(y, y_lazy, zero_division=0),
    "F1":        f1_score(y, y_lazy, zero_division=0),
}).round(3)
baseline.to_frame("everything = false positive")

,everything = false positive
accuracy,0.638
precision,0.000
recall,0.000
F1,0.000


## 6. Split by star (60 / 20 / 20)
Several KOIs can orbit the same star and share all its star values. If one went to training and another to test, the model could recognise the star instead of learning the physics, and the test score would be too optimistic (R4). So whole stars go to one set.

`GroupShuffleSplit` first takes 20 % of the stars for **test**, then 25 % of the remaining stars for **validation** (0.25 × 0.8 = 0.2), leaving 60 % for **train**. The fractions are of stars, so row percentages are close to, not exactly, 60/20/20.

In [22]:
groups = lab["kepid"]

gss_test = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
trval_idx, test_idx = next(gss_test.split(lab, groups=groups))

trval = lab.iloc[trval_idx]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
tr_rel, va_rel = next(gss_val.split(trval, groups=trval["kepid"]))

lab["split"] = "test"
lab.loc[trval.index[tr_rel], "split"] = "train"
lab.loc[trval.index[va_rel], "split"] = "val"
lab["split"].value_counts()

split
train    4519
test     1546
val      1522
Name: count, dtype: int64

In [23]:
# Check 1: no star appears in two sets
stars = {s: set(lab.loc[lab["split"] == s, "kepid"]) for s in ["train", "val", "test"]}
assert stars["train"].isdisjoint(stars["val"]),  "a star is in both train and val"
assert stars["train"].isdisjoint(stars["test"]), "a star is in both train and test"
assert stars["val"].isdisjoint(stars["test"]),   "a star is in both val and test"
print("OK: no kepid appears in more than one set.")

# Check 2: sizes and planet rate are similar in all three sets
split_summary = (lab.groupby("split")
                    .agg(rows=("label", "size"), stars=("kepid", "nunique"), planet_rate=("label", "mean"))
                    .reindex(["train", "val", "test"]))
split_summary["% of rows"] = (100 * split_summary["rows"] / len(lab)).round(1)
split_summary["planet_rate"] = split_summary["planet_rate"].round(3)
split_summary

OK: no kepid appears in more than one set.


,rows,stars,planet_rate,% of rows
split,,,,
train,4519,3984,0.352,59.6
val,1522,1328,0.376,20.1
test,1546,1329,0.380,20.4


## 7. Save
`koi_split.csv` keeps **all** columns plus `label` and `split`. The leaky columns (`koi_score`, `koi_fpflag_*`, `koi_pdisposition`, `kepler_name`, `koi_count`) stay in the file because P6 uses the flags for the failure analysis, but `features.py` (P2) must never use them as inputs.

In [24]:
out = PROCESSED / "koi_split.csv"
lab.to_csv(out, index=False)
print(f"Saved {out.relative_to(ROOT)}  ->  {lab.shape[0]} rows, {lab.shape[1]} columns")
print("Next: commit and push (Source Control → message 'P1: data audit + split' → Commit → Sync Changes).")

Saved data\processed\koi_split.csv  ->  7587 rows, 155 columns
Next: commit and push (Source Control → message 'P1: data audit + split' → Commit → Sync Changes).


## Explain (for the report / professor)
*"Several detections can orbit the same star and share its values, so we split by star; otherwise the test score would be too optimistic (R4)."*